
## Folder 03 — Bronze Ingestion with Auto Loader:


**Ingest all seven CSV datasets from the landing directory into Bronze Delta tables using Databricks Auto Loader.**


### Bronze Ingestion Architecture:


```text
┌────────────────────────────────────────────────────┐
│                    LANDING LAYER                   │
│                                                    │
│                7 Original CSV Files                │
│                                                    │
│  customers.csv             products.csv            │
│  orders.csv                order_items.csv         │
│  payments.csv              shipments.csv           │
│  inventory_snapshots.csv                           │
└────────────────────────────────────────────────────┘
                          │
                          ▼
┌────────────────────────────────────────────────────┐
│                     AUTO LOADER                    │
│                                                    │
│  • Schema Inference                                │
│  • Incremental File Discovery                      │
│  • Checkpoint Management                           │
│  • Schema Evolution                                │
└────────────────────────────────────────────────────┘
                          │
                          ▼
┌────────────────────────────────────────────────────┐
│                 BRONZE LAYER - DELTA               │
│                                                    │
│  • Raw Source Records                              │
│  • Ingestion Metadata                              │
│  • Source File Information                         │
│  • Ingestion Timestamp                             │
└────────────────────────────────────────────────────┘
                          │
                          ▼
┌────────────────────────────────────────────────────┐
│                 BRONZE DELTA TABLES                │
│                                                    │
│  1. bronze.customers                               │
│  2. bronze.products                                │
│  3. bronze.orders                                  │
│  4. bronze.order_items                             │
│  5. bronze.payments                                │
│  6. bronze.shipments                               │
│  7. bronze.inventory_snapshots                     │
└────────────────────────────────────────────────────┘
```

In [0]:

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BASE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze"

SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze"

BRONZE_SCHEMA = "ecommerce_bronze"

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

print("Project configuration completed.")


# ============================================================
# 3. CREATE BRONZE SCHEMA
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}
    """
)

print("Bronze schema ready.")


# ============================================================
# 4. BRONZE INGESTION FUNCTION
# ============================================================

def ingest_bronze(dataset):

    source_path = f"{LANDING_PATH}/{dataset}"
    checkpoint_path = f"{CHECKPOINT_PATH}/{dataset}"
    schema_path = f"{SCHEMA_PATH}/{dataset}"

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    print("\n" + "=" * 70)
    print(f"BRONZE INGESTION: {dataset.upper()}")
    print("=" * 70)

    # --------------------------------------------------------
    # 4.1 VALIDATE SOURCE FILES
    # --------------------------------------------------------

    source_files = [
        file
        for file in dbutils.fs.ls(source_path)
        if file.path.lower().endswith(".csv")
    ]

    if not source_files:
        raise ValueError(
            f"No CSV files found: {source_path}"
        )

    print(f"CSV files found: {len(source_files)}")

    # --------------------------------------------------------
    # 4.2 READ USING AUTO LOADER
    # --------------------------------------------------------

    source_df = (

        spark.readStream

        .format("cloudFiles")

        .option("cloudFiles.format", "csv")

        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )

        .option(
            "cloudFiles.inferColumnTypes",
            "true"
        )

        .option("header", "true")

        .option("mode", "PERMISSIVE")

        .load(source_path)

    )

    # --------------------------------------------------------
    # 4.3 ADD INGESTION METADATA
    # --------------------------------------------------------

    bronze_df = (

        source_df

        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )

        .withColumn(
            "_ingestion_date",
            F.current_date()
        )

    )

    # --------------------------------------------------------
    # 4.4 WRITE TO UNITY CATALOG DELTA TABLE
    # --------------------------------------------------------

    query = (

        bronze_df

        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            checkpoint_path
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(table_name)

    )

    query.awaitTermination()

    print("Auto Loader ingestion completed.")

    # --------------------------------------------------------
    # 4.5 READ BRONZE TABLE
    # --------------------------------------------------------

    result_df = spark.table(table_name)

    record_count = result_df.count()

    if record_count == 0:
        raise ValueError(
            f"No records found in {table_name}"
        )

    # --------------------------------------------------------
    # 4.6 VALIDATE METADATA
    # --------------------------------------------------------

    required_columns = [
        "_source_file",
        "_ingestion_date"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in result_df.columns
    ]

    # Support the established timestamp column and
    # the older column name for compatibility.

    timestamp_candidates = [
        "_ingestion_timestamp",
        "_ingested_at"
    ]

    available_timestamp = next(
        (
            column
            for column in timestamp_candidates
            if column in result_df.columns
        ),
        None
    )

    if available_timestamp is None:
        missing_columns.append(
            "ingestion timestamp "
            "(_ingestion_timestamp or _ingested_at)"
        )

    if missing_columns:
        raise ValueError(
            f"Missing metadata columns: {missing_columns}"
        )

    # --------------------------------------------------------
    # 4.7 DISPLAY RESULTS
    # --------------------------------------------------------

    print("\nINGESTION SUCCESSFUL")

    print(f"Dataset           : {dataset}")
    print(f"Record Count      : {record_count}")
    print(f"Table             : {table_name}")
    print(f"Timestamp Column  : {available_timestamp}")

    display(result_df.limit(5))

    return {

        "dataset": dataset,
        "records": record_count,
        "table_name": table_name,
        "status": "SUCCESS",
        "error": ""

    }


# ============================================================
# 5. PROCESS ALL DATASETS
# ============================================================

ingestion_results = []

for dataset in DATASETS:

    try:

        result = ingest_bronze(dataset)

        ingestion_results.append(result)

    except Exception as e:

        error_message = str(e)[:2000]

        print("\n" + "!" * 70)
        print(f"FAILED DATASET: {dataset}")
        print(error_message)
        print("!" * 70)

        ingestion_results.append({

            "dataset": dataset,
            "records": 0,
            "table_name": (
                f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
            ),
            "status": "FAILED",
            "error": error_message

        })


# ============================================================
# 6. INGESTION SUMMARY
# ============================================================

summary_df = spark.createDataFrame(
    ingestion_results
)

display(
    summary_df.orderBy("dataset")
)


# ============================================================
# 7. EXECUTION STATISTICS
# ============================================================

success_count = sum(
    1
    for row in ingestion_results
    if row["status"] == "SUCCESS"
)

failed_count = sum(
    1
    for row in ingestion_results
    if row["status"] == "FAILED"
)

total_records = sum(
    row["records"]
    for row in ingestion_results
)

print("\n" + "=" * 70)
print("BRONZE INGESTION SUMMARY")
print("=" * 70)

print(f"Total datasets : {len(DATASETS)}")
print(f"Successful     : {success_count}")
print(f"Failed         : {failed_count}")
print(f"Total records  : {total_records}")


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

if failed_count > 0:

    failed_datasets = [
        row["dataset"]
        for row in ingestion_results
        if row["status"] == "FAILED"
    ]

    raise RuntimeError(
        f"Bronze ingestion failed: {failed_datasets}"
    )


print("\n" + "=" * 70)
print("ALL 7 BRONZE DATASETS PROCESSED SUCCESSFULLY")
print("=" * 70)


In [0]:
for dataset in DATASETS:

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    )

    exists = spark.catalog.tableExists(table_name)

    print(table_name, "EXISTS" if exists else "NOT FOUND")


**Check that all 7 Bronze tables exist and contain records:**

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_bronze;


**Then verify row counts:**

In [0]:
%sql
SELECT 'customers' AS dataset, COUNT(*) AS total_records
FROM workspace.ecommerce_bronze.bronze_customers

UNION ALL

SELECT 'products', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_products

UNION ALL

SELECT 'orders', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_payments

UNION ALL

SELECT 'shipments', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_shipments

UNION ALL

SELECT 'inventory_snapshots', COUNT(*)
FROM workspace.ecommerce_bronze.bronze_inventory_snapshots;